## Quantum Sand Project Template SQLAlchemy 001

### Import dependencies

We will import `create_engine` from `sqlalchemy`, `pandas` and `geopandas`.

In [1]:
from sqlalchemy import create_engine
import pandas as pd
import geopandas as gpd

### Running Quantum Sand Rails locally

If you are just looking at this notebook statically in a browser or VS Code you can just read this notebook without running the Python code.

Having the Quantum Sand Rails; Ruby-on-Rails PostGIS API; running locally is necessary for running the Python code later in this notebook.

The `seeds.rb` within Quantum Sand Rails has some test data which we can use for this example;

This block of Ruby code imports some seed data from `geospatial_traces = []` into the `geospatial_traces` table;

```ruby
geospatial_traces = [
  {
    name: "Harbour seals in Christchurch Bay",
    geospatial: "POINT(50.713294760937224 -1.6601009519012668)",
    data: {
      name: "Harbour seal",
      scientific_name: "Phoca vitulina"
    }
  }
]

geospatial_traces.each do |geospatial_trace|
  GeospatialTrace.where(name: geospatial_trace[:name]).first_or_create do |trace|
    trace.name = geospatial_trace[:name]
    trace.geospatial = geospatial_trace[:geospatial]
    trace.data = geospatial_trace[:data]

    puts "Created geospatial_trace with name: #{geospatial_trace[:name]}"
  end
end
```

The `geospatial_traces` have the following attributes; the columns in the database table;

* `name` is a string, `"Harbour seals in Christchurch Bay"`.
* `geospatial` is PostGIS geometry, `"POINT(50.713294760937224 -1.6601009519012668)"`.
* `data` is JSON, `{name: "Harbour seal", scientific_name: "Phoca vitulina"}.`

We will use Python with SQLAlchemy (and Psycopg2) to obtain a new `Engine` instance. We can then use this `engine` with;

* `read_sql_query()` from pandas.
* `read_postgis()` from GeoPandas.

To load the data from the `geospatial_traces` database table within the Rails app.

### Access the Rails PostGIS PostgreSQL database using SQLAlchemy (and Psycopg2)

If you are just looking at this notebook statically in a browser or VS Code you can just read the outcome of this section.

You need to run Quantum Sand Rails locally if you want to run this Python code.

We do not need to explicitly `import psycopg2`. However this notebook will fail to run unless Psycopg2 is installed.

We can define `rails_postgis_postgres_url` as `"postgresql://127.0.0.1/quantumsand_rails_development"`.

To explain this url;

* `postgresql://` indicates that we are connecting to a PostgreSQL database.
* `username:password`, which are both deliberately missing.
* `@hostname` is `@127.0.0.1`, but in this scenario we can drop the `@`, simplified just to `127.0.0.1`.
* `database_name`, which is `quantumsand_rails_development`.

The `username` and `password` are deliberately blank because Quantum Sand Rails has been locally configured to not require authentication.

`127.0.0.1` is the standard IP address for a computer or device to communicate directly with itself.

Using `create_engine()` within SQLAlchemy with the argument;

* `url` which is `rails_postgis_postgres_url`.

Creates a new `Engine` instance into `engine`.

We can define the `sql_query` as `"select * from geospatial_traces limit 1"`;

```sql
select * from geospatial_traces limit 1
```

With PostgreSQL we could also express this as;

```sql
SELECT * FROM geospatial_traces LIMIT 1
```

Or even like this;

```sql
SELECT *
FROM geospatial_traces
LIMIT 1
```

Using `read_sql_query()` from pandas with the arguments;

* `sql` is the sql query to execute, `sql_query`.
* `con` is the SQLAlchemy connectable, `engine`.

We can return a pandas DataFrame `pandas_dataframe` corresponding to the result set of the sql query string.

The result of `print()` for `pandas_dataframe` reveals all of the correct data from the `geospatial_traces` table.

However, it is worth noticing that the `geospatial` column within `pandas_dataframe` outputs `"0101000020E61000001751233E4D5B49402F970704C68F..."`.

If we use `info()` from pandas DataFrame we can see that the `Dtype` for the `geospatial` column within `pandas_dataframe` is actually `str`, a string.

If we now attempt the same using `read_postgis()` from GeoPandas with the arguments;

* `sql` is the sql query to execute, `sql_query`.
* `con` is the SQLAlchemy connectable, `engine`.
* `geom_col` is the column name to convert to shapely geometries, `"geospatial"`.

We can return a GeoPandas GeoDataFrame `geopandas_geodataframe` corresponding to the result set of the sql query string.

The result of `print()` for `geopandas_geodataframe` reveals all of the correct data from the `geospatial_traces` table.

However, this time, it is worth noticing that the `geospatial` column within `geopandas_geodataframe` outputs the correct `"POINT (50.71329 -1.6601)"`.

If we use `info()` from GeoPandas GeoDataFrame we can see that the `Dtype` for the `geospatial` column within `geopandas_geodataframe` is actually `geometry` this time, rather than a string.

Finally the line `geopandas_geodataframe` outputs a nicely formatted table.

In [2]:
rails_postgis_postgres_url = "postgresql://127.0.0.1/quantumsand_rails_development"

engine = create_engine(rails_postgis_postgres_url)

sql_query = "select * from geospatial_traces limit 1"
print(sql_query)

print("print() pandas dataframe read_sql_query()")
pandas_dataframe = pd.read_sql_query(sql_query, engine)
print(pandas_dataframe)
print()

print("print() pandas dataframe info()")
pandas_dataframe_info = pandas_dataframe.info()
print(pandas_dataframe_info)
print()

print("print() GeoPandas GeoDataFrame read_postgis()")
geopandas_geodataframe = gpd.read_postgis(sql_query, engine, geom_col="geospatial")
print(geopandas_geodataframe)
print()

print("print() GeoPandas GeoDataFrame info()")
geopandas_geodataframe_info = geopandas_geodataframe.info()
print(geopandas_geodataframe_info)
print()

geopandas_geodataframe

select * from geospatial_traces limit 1
print() pandas dataframe read_sql_query()
   id                               name  \
0   1  Harbour seals in Christchurch Bay   

                                          geospatial  \
0  0101000020E61000001751233E4D5B49402F970704C68F...   

                                                data  \
0  {'name': 'Harbour seal', 'scientific_name': 'P...   

                  created_at                 updated_at  
0 2026-06-19 15:04:34.602421 2026-06-19 15:04:34.602421  

print() pandas dataframe info()
<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id          1 non-null      int64         
 1   name        1 non-null      str           
 2   geospatial  1 non-null      str           
 3   data        1 non-null      object        
 4   created_at  1 non-null      datetime64[us]
 5   updated_at  1 non-null  

,id,name,geospatial,data,created_at,updated_at
0,1,Harbour seals in Christchurch Bay,POINT (50.71329 -1.6601),"{'name': 'Harbour seal', 'scientific_name': 'P...",2026-06-19 15:04:34.602421,2026-06-19 15:04:34.602421


You can see the same data that was used to seed the `geospatial_traces` table.

More to follow...